# EDAノートブック

本ノートブックは、分析業務で使うEDAを固定手順で実行するための定型版です。
可視化結果は相対パスで `reports/figures` に保存します。


## 固定EDA計画
1. データ読み込みと基本確認
2. 列型・記述統計の確認
3. 欠損率の集計と可視化
4. 数値列の分布確認
5. カテゴリ列の主要分布確認
6. 目的変数の分布と偏り確認
7. 数値特徴量の相関確認
8. 日付列の時系列傾向確認（存在時）
9. 観察結果サマリ


In [ ]:
from pathlib import Path
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")
warnings.filterwarnings("ignore", message="Glyph .* missing from font")
plt.rcParams["axes.unicode_minus"] = False

JP_FONT_CANDIDATES = [
    "Yu Gothic",
    "Meiryo",
    "MS Gothic",
    "Noto Sans CJK JP",
    "IPAexGothic",
    "IPAGothic",
    "TakaoGothic",
]


def configure_japanese_font() -> str:
    available = {f.name for f in fm.fontManager.ttflist}
    for name in JP_FONT_CANDIDATES:
        if name in available:
            plt.rcParams["font.family"] = name
            return name
    plt.rcParams["font.sans-serif"] = JP_FONT_CANDIDATES + list(plt.rcParams.get("font.sans-serif", []))
    return ""


selected_font = configure_japanese_font()
if selected_font:
    pass
else:
    pass

ENCODINGS = ("utf-8-sig", "utf-8", "cp932", "shift_jis", "euc_jp")

cwd = Path.cwd()
if (cwd / "configs" / "project_config.json").exists():
    analysis_root = cwd
elif (cwd.name == "notebooks") and (cwd.parent / "configs" / "project_config.json").exists():
    analysis_root = cwd.parent
elif (cwd / "artifacts" / "analysis_project" / "configs" / "project_config.json").exists():
    analysis_root = cwd / "artifacts" / "analysis_project"
else:
    analysis_root = cwd

FIG_DIR = analysis_root / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_csv_auto(path: Path):
    for enc in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise RuntimeError("CSVを既知エンコーディングで読み込めませんでした")


def is_pure_day_number_column(series: pd.Series) -> bool:
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return False
    if not np.all(np.isclose(numeric, np.round(numeric))):
        return False
    return bool(((numeric >= 1) & (numeric <= 31)).all())


In [ ]:
csv_rel = Path("data/train.csv")
candidates = [
    analysis_root / csv_rel,
    Path("data/train.csv"),
    Path("artifacts/analysis_project") / csv_rel,
]
csv_path = next((p for p in candidates if p.exists()), candidates[0])
df, used_encoding = load_csv_auto(csv_path)
target_col = "y"
if target_col not in df.columns:
    target_col = df.columns[-1]
date_col_hint = "day".strip() or None



## 1. データ概要


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) > 0:
        pass
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    if len(cat_cols) > 0:
        cat_summary = pd.DataFrame({
            '欠損数': df[cat_cols].isnull().sum(),
            'ユニーク数': df[cat_cols].nunique(dropna=True),
            '最頻値': [df[c].mode(dropna=True).iloc[0] if not df[c].mode(dropna=True).empty else np.nan for c in cat_cols]
        })
except Exception as _eda_exc:
    dtype_summary = (
        df.dtypes.astype(str)
        .rename("dtype")
        .reset_index()
        .rename(columns={"index": "column"})
    )
    type_counts = dtype_summary["dtype"].value_counts().rename_axis("dtype").reset_index(name="count")
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    category_cols = [c for c in df.columns if c not in numeric_cols]


## 2. 欠損分析


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    missing_rate = (df.isnull().mean() * 100).sort_values(ascending=False)
    missing_df = missing_rate.reset_index()
    missing_df.columns = ['column', 'missing_rate']

    plot_df = missing_df.head(20)
    plt.figure(figsize=(10, max(4, 0.4 * len(plot_df))))
    sns.barplot(data=plot_df, y='column', x='missing_rate', palette='Blues_r')
    plt.title('欠損率 上位20項目')
    plt.xlabel('欠損率（%）')
    plt.ylabel('列名')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'missing_rate_top20.png', dpi=150, bbox_inches='tight')
    plt.close()
except Exception as _eda_exc:
    missing_rate = (df.isna().mean() * 100).sort_values(ascending=False)
    top_missing = missing_rate.head(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    top_missing.plot(kind="bar", ax=ax, color="#4c78a8")
    ax.set_title("欠損率 上位20列")
    ax.set_ylabel("欠損率(%)")
    ax.set_xlabel("列名")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "missing_rate_top20.png", dpi=160, bbox_inches="tight")


## 3. 数値特徴量の分布


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in num_cols:
        num_cols_wo_target = [c for c in num_cols if c != target_col]
    else:
        num_cols_wo_target = num_cols.copy()

    if 'id' in num_cols_wo_target:
        num_cols_wo_target.remove('id')

    stats_df = pd.DataFrame({
        'dtype': df[num_cols].dtypes.astype(str),
        '欠損数': df[num_cols].isnull().sum(),
        '平均': df[num_cols].mean(numeric_only=True),
        '中央値': df[num_cols].median(numeric_only=True),
        '標準偏差': df[num_cols].std(numeric_only=True),
        '最小値': df[num_cols].min(numeric_only=True),
        '最大値': df[num_cols].max(numeric_only=True)
    }) if len(num_cols) > 0 else pd.DataFrame()

    plot_cols = num_cols_wo_target[:6]
    if len(plot_cols) > 0:
        n = len(plot_cols)
        rows = int(np.ceil(n / 2))
        fig, axes = plt.subplots(rows, 2, figsize=(14, 4 * rows))
        axes = np.array(axes).reshape(-1)
        for ax, col in zip(axes, plot_cols):
            series = df[col].dropna()
            sns.histplot(series, kde=True, ax=ax, color='steelblue')
            ax.set_title(f'{col} の分布')
            ax.set_xlabel(col)
            ax.set_ylabel('件数')
        for ax in axes[n:]:
            ax.axis('off')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'numeric_distribution_top6.png', dpi=150, bbox_inches='tight')
        plt.close()
except Exception as _eda_exc:
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if numeric_cols:
        pass
    target_col_local = "y"
    if target_col_local not in df.columns:
        target_col_local = df.columns[-1]
    plot_cols = [c for c in numeric_cols if c != target_col_local][:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(plot_cols):
            col = plot_cols[i]
            sns.histplot(df[col], bins=30, ax=ax, color="#1f77b4")
            ax.set_title(f"{col} の分布")
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "numeric_distribution_top6.png", dpi=160, bbox_inches="tight")


## 4. カテゴリ特徴量の分布


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    summary_rows = []
    for col in cat_cols:
        vc = df[col].astype('object').fillna('欠損').value_counts()
        top_val = vc.index[0] if len(vc) > 0 else np.nan
        top_cnt = vc.iloc[0] if len(vc) > 0 else np.nan
        summary_rows.append({
            '列名': col,
            'ユニーク数': df[col].nunique(dropna=True),
            '最頻値': top_val,
            '最頻値件数': top_cnt
        })
    cat_summary = pd.DataFrame(summary_rows).sort_values(['ユニーク数', '列名']) if len(summary_rows) > 0 else pd.DataFrame()

    plot_cols = [c for c in cat_cols if c != target_col][:3]
    if len(plot_cols) > 0:
        fig, axes = plt.subplots(len(plot_cols), 1, figsize=(12, 4 * len(plot_cols)))
        if len(plot_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, plot_cols):
            vc = df[col].astype('object').fillna('欠損').value_counts().head(15)
            sns.barplot(x=vc.values, y=vc.index, ax=ax, palette='viridis')
            ax.set_title(f'{col} の分布（上位15件）')
            ax.set_xlabel('件数')
            ax.set_ylabel(col)
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'categorical_distribution_top3.png', dpi=150, bbox_inches='tight')
        plt.close()
except Exception as _eda_exc:
    category_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    plot_cols = category_cols[:3]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for i, ax in enumerate(axes):
        if i < len(plot_cols):
            col = plot_cols[i]
            vc = df[col].astype(str).fillna("欠損").value_counts().head(10)
            vc.plot(kind="bar", ax=ax, color="#59a14f")
            ax.set_title(f"{col} 上位カテゴリ")
            ax.tick_params(axis="x", rotation=45)
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "categorical_distribution_top3.png", dpi=160, bbox_inches="tight")


## 5. 目的変数分析


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns


    plt.figure(figsize=(6, 4))
    order = sorted(df[target_col].dropna().unique().tolist()) if df[target_col].dropna().nunique() > 0 else None
    ax = sns.countplot(data=df, x=target_col, order=order, palette='Set2')
    plt.title('目的変数の分布')
    plt.xlabel('目的変数')
    plt.ylabel('件数')
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f'{int(h)}', (p.get_x() + p.get_width() / 2, h), ha='center', va='bottom')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
except Exception as _eda_exc:
    series = df[target_col]
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    if pd.api.types.is_numeric_dtype(series):
        uniq = series.dropna().nunique()
        if uniq > 20:
            sns.histplot(series.dropna(), bins=30, ax=ax[0], color="#f28e2b")
            ax[0].set_title("目的変数ヒストグラム")
            sns.boxplot(x=series.dropna(), ax=ax[1], color="#e15759")
            ax[1].set_title("目的変数ボックスプロット")
        else:
            vc = series.value_counts(dropna=False).sort_index()
            vc.plot(kind="bar", ax=ax[0], color="#f28e2b")
            ax[0].set_title("目的変数カテゴリ分布")
            (vc / vc.sum() * 100).round(2).plot(kind="bar", ax=ax[1], color="#e15759")
            ax[1].set_title("目的変数カテゴリ比率(%)")
    else:
        vc = series.astype(str).fillna("欠損").value_counts().head(20)
        vc.plot(kind="bar", ax=ax[0], color="#f28e2b")
        ax[0].set_title("目的変数カテゴリ分布")
        (vc / vc.sum() * 100).round(2).plot(kind="bar", ax=ax[1], color="#e15759")
        ax[1].set_title("目的変数カテゴリ比率(%)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "target_distribution.png", dpi=160, bbox_inches="tight")


## 6. 相関分析


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) >= 2:
        corr = df[num_cols].corr(numeric_only=True)
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, square=False)
        plt.title('数値特徴量の相関ヒートマップ')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'feature_correlation_heatmap.png', dpi=150, bbox_inches='tight')
        plt.close()

        if target_col in corr.columns:
            target_corr = corr[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)
except Exception as _eda_exc:
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    fig, ax = plt.subplots(figsize=(10, 8))
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols[:20]].corr(numeric_only=True)
        sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
        ax.set_title("数値特徴量の相関ヒートマップ（先頭20列）")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "相関分析に十分な数値列がありません", ha="center", va="center", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "feature_correlation_heatmap.png", dpi=160, bbox_inches="tight")


## 7. 日付分析


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    def is_pure_day_number_column(series):
        s = pd.to_numeric(series, errors='coerce')
        valid = s.dropna()
        if len(valid) == 0:
            return False
        return valid.between(1, 31).all()

    used_col = None
    if date_col_hint is not None and date_col_hint in df.columns and is_pure_day_number_column(df[date_col_hint]):
        used_col = date_col_hint
    elif 'day' in df.columns and is_pure_day_number_column(df['day']):
        used_col = 'day'
    else:
        parsed_candidates = []
        for col in df.columns:
            if col == target_col:
                continue
            if df[col].dtype == 'object':
                parsed = pd.to_datetime(df[col], errors='coerce')
                if parsed.notna().mean() >= 0.8:
                    parsed_candidates.append((col, parsed))
        if len(parsed_candidates) > 0:
            used_col, parsed_series = parsed_candidates[0]

    if used_col is not None:
        if is_pure_day_number_column(df[used_col]):
            tmp = df[[used_col, target_col]].copy()
            tmp[used_col] = pd.to_numeric(tmp[used_col], errors='coerce')
            agg = tmp.groupby(used_col).agg(件数=(target_col, 'size'))
            if pd.api.types.is_numeric_dtype(tmp[target_col]):
                agg['目的変数平均'] = tmp.groupby(used_col)[target_col].mean()
            fig, ax1 = plt.subplots(figsize=(10, 5))
            ax1.plot(agg.index, agg['件数'], marker='o', color='tab:blue')
            ax1.set_title(f'{used_col} による件数推移')
            ax1.set_xlabel('日')
            ax1.set_ylabel('件数', color='tab:blue')
            ax1.tick_params(axis='y', labelcolor='tab:blue')
            if '目的変数平均' in agg.columns:
                ax2 = ax1.twinx()
                ax2.plot(agg.index, agg['目的変数平均'], marker='s', color='tab:orange')
                ax2.set_ylabel('目的変数平均', color='tab:orange')
                ax2.tick_params(axis='y', labelcolor='tab:orange')
            plt.tight_layout()
            plt.savefig(FIG_DIR / 'figure_06.png', dpi=150, bbox_inches='tight')
            plt.close()
        else:
            parsed = pd.to_datetime(df[used_col], errors='coerce')
            tmp = pd.DataFrame({'date': parsed, target_col: df[target_col]}).dropna(subset=['date'])
            tmp['date_key'] = tmp['date'].dt.to_period('M').astype(str)
            agg = tmp.groupby('date_key').agg(件数=(target_col, 'size'))
            if pd.api.types.is_numeric_dtype(tmp[target_col]):
                agg['目的変数平均'] = tmp.groupby('date_key')[target_col].mean()
            fig, ax1 = plt.subplots(figsize=(12, 5))
            ax1.plot(range(len(agg)), agg['件数'], marker='o', color='tab:blue')
            ax1.set_title(f'{used_col} の時系列推移')
            ax1.set_xlabel('時点')
            ax1.set_ylabel('件数', color='tab:blue')
            ax1.set_xticks(range(len(agg)))
            ax1.set_xticklabels(agg.index, rotation=45, ha='right')
            ax1.tick_params(axis='y', labelcolor='tab:blue')
            if '目的変数平均' in agg.columns:
                ax2 = ax1.twinx()
                ax2.plot(range(len(agg)), agg['目的変数平均'], marker='s', color='tab:orange')
                ax2.set_ylabel('目的変数平均', color='tab:orange')
                ax2.tick_params(axis='y', labelcolor='tab:orange')
            plt.tight_layout()
            plt.savefig(FIG_DIR / 'figure_06.png', dpi=150, bbox_inches='tight')
            plt.close()
except Exception as _eda_exc:
    date_col = "day".strip() or date_col_hint
    fig, ax = plt.subplots(figsize=(12, 4))
    if date_col and date_col in df.columns and date_col != target_col:
        pure_day = is_pure_day_number_column(df[date_col])
        if pure_day:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{date_col} は純粋な日番号列のため日付展開を行いません", ha="center", va="center", fontsize=12)
        else:
            parsed = pd.to_datetime(df[date_col], errors="coerce")
            valid = parsed.notna()
            if valid.sum() > 0:
                tmp = df.loc[valid, [target_col]].copy()
                tmp["_date"] = parsed.loc[valid]
                if pd.api.types.is_numeric_dtype(tmp[target_col]):
                    monthly = tmp.set_index("_date")[target_col].resample("M").mean()
                    monthly.plot(ax=ax, color="#4e79a7", marker="o")
                    ax.set_title("月次の目的変数平均")
                    ax.set_ylabel("平均値")
                else:
                    monthly = tmp.assign(_value=1).set_index("_date")["_value"].resample("M").sum()
                    monthly.plot(ax=ax, color="#4e79a7", marker="o")
                    ax.set_title("月次レコード件数")
                    ax.set_ylabel("件数")
                ax.set_xlabel("日付")
            else:
                ax.axis("off")
                ax.text(0.5, 0.5, f"{date_col} を日付として解釈できませんでした", ha="center", va="center", fontsize=12)
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "日付分析対象列はありません", ha="center", va="center", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "figure_06.png", dpi=160, bbox_inches="tight")


## 8. 観察結果サマリ


In [ ]:
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    n_rows, n_cols = df.shape
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    missing_total = int(df.isnull().sum().sum())
    missing_by_col = df.isnull().sum().sort_values(ascending=False)


    if target_col in df.columns:
        pass

    if len(num_cols) > 1 and target_col in num_cols:
        corr_s = df[num_cols].corr(numeric_only=True)[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)

    high_card = []
    for c in cat_cols:
        high_card.append({'列名': c, 'ユニーク数': df[c].nunique(dropna=True)})
    if len(high_card) > 0:
        high_card_df = pd.DataFrame(high_card).sort_values('ユニーク数', ascending=False)
except Exception as _eda_exc:
    summary_rows = []
    summary_rows.append(f"レコード数: {len(df):,}")
    summary_rows.append(f"列数: {df.shape[1]:,}")
    summary_rows.append(f"欠損率上位列: {', '.join((df.isna().mean()*100).sort_values(ascending=False).head(3).index.tolist())}")
    summary_rows.append(f"数値列数: {len(df.select_dtypes(include=['number']).columns)}")
    summary_rows.append(f"カテゴリ列数: {len([c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])])}")
    summary_rows.append(f"目的変数候補: {target_col}")
    for row in summary_rows:
        pass
